# CS781 Phase 6 — Abstractive Legal Summarization
Every experiment starts from the Phase 1 LegalLM (decoder-only, 12L / 768d, ctx 1024).

**Long-document strategy (chunk-level summarization with alignment):** each judgment is split into
sentence-aligned chunks of <= 600 tokens. Every reference-summary sentence is assigned to the chunk it
overlaps most (unigram + bigram overlap), giving (chunk -> partial summary) training pairs that fit in the
1024-token window: `Part i of n:` + chunk + `Summary:` + target + EOS. At test time every chunk is summarised
(batched greedy decoding with KV cache, 3-gram blocking, repetition penalty), pieces are stitched in
document order and near-duplicate sentences are removed.

| Submission | Technique |
|---|---|
| 01_fullft_chunk | Full fine-tuning (reference + KD teacher) |
| 02_lora_chunk | LoRA on qkv/proj/w12/w3 (compulsory) |
| 03_houlsby_adapter | Houlsby adapters (after attention and after FFN) |
| 04_pfeiffer_adapter | Pfeiffer adapters (after FFN only) |
| 05_prefix_tuning | Prefix tuning (learned per-layer key/value prefixes, MLP reparameterised) |
| 06_kd_student6 | Knowledge distillation: 12L teacher -> 6L student (token-level KL on teacher logits + CE) |
| 07_lora_lead | LoRA, flat baseline: first 600 tokens -> first 400 summary tokens (no chunking) |
| 08_mbr_consensus | Minimum-Bayes-risk selection: per document, the candidate summary that agrees most with the others |

In [ ]:
# Config
import os, sys, json, math, time, random, re, shutil, zipfile, gc, csv
from collections import Counter
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("SUMM_SMOKE", "0") == "1"   # set True for a tiny dry run
USE_BERTSCORE = False   # True = also compute BERTScore locally (downloads roberta-large; evaluation only)
RUN_START = time.time()
INPUT_ROOT = os.environ.get("SUMM_INPUT", "/kaggle/input")
OUT = os.environ.get("SUMM_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)

CFG = dict(
    seed=42,
    chunk_tokens=600,      # document tokens per chunk
    target_max=400,        # max summary tokens per chunk (prompt + target + EOS <= 1024)
    max_chunks_train=40,   # cap on chunk pairs per training document (evenly spread)
    max_chunks_infer=40,   # cap on chunks summarised per test document
    keep_empty_frac=0.3,   # fraction of chunks with no aligned summary sentence kept as (chunk -> EOS) pairs
    micro=8, accum=4,      # sequences per forward / steps of accumulation -> 32 sequences per update
    val_docs=300,          # held-out training documents (no public dev split)
    val_gen_docs=80,       # of which this many are fully summarised for ROUGE
    val_loss_pairs=400,    # chunk pairs used for validation loss (checkpoint selection)
    n_evals=3,
    gen_batch=48, max_new=360, no_repeat_ngram=3, rep_penalty=1.15,
    total_hours=4.6,
)

EXPS = [
    dict(name="01_fullft_chunk",     method="full",     lr=5e-5, epochs=2, minutes=35),
    dict(name="06_kd_student6",      method="kd",       lr=1e-4, epochs=2, minutes=28,
         layers=[0, 2, 4, 7, 9, 11], T=2.0, alpha=0.5),
    dict(name="02_lora_chunk",       method="lora",     lr=3e-4, epochs=2, minutes=28, r=16, alpha=32),
    dict(name="03_houlsby_adapter",  method="houlsby",  lr=1e-3, epochs=2, minutes=25, bottleneck=64),
    dict(name="04_pfeiffer_adapter", method="pfeiffer", lr=1e-3, epochs=2, minutes=22, bottleneck=96),
    dict(name="05_prefix_tuning",    method="prefix",   lr=3e-3, epochs=2, minutes=22, n_prefix=30),
    dict(name="07_lora_lead",        method="lora",     lr=3e-4, epochs=3, minutes=12, r=16, alpha=32,
         mode="lead", optional=True),
]

if SMOKE:
    CFG.update(chunk_tokens=96, target_max=48, max_chunks_train=4, max_chunks_infer=4, micro=2, accum=2,
               val_docs=6, val_gen_docs=3, val_loss_pairs=8, n_evals=2, gen_batch=8, max_new=12, total_hours=1.0)
    for e in EXPS:
        e.update(epochs=1, minutes=0.3)
        if e["method"] == "prefix":
            e["n_prefix"] = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("device:", device, torch.cuda.get_device_name(0) if USE_AMP else "")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_all(CFG["seed"])


def elapsed_h():
    return (time.time() - RUN_START) / 3600


def free():
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()

In [ ]:
# Locate inputs (searches every attached dataset and explains what it skips)
def find_files(name):
    hits = []
    for root, _, files in os.walk(INPUT_ROOT, followlinks=True):
        if name in files:
            hits.append(os.path.join(root, name))
    return sorted(hits)


def read_any_json(p):
    raw = open(p, encoding="utf-8").read()
    try:
        d = json.loads(raw)
    except json.JSONDecodeError:
        d = [json.loads(l) for l in raw.splitlines() if l.strip()]
    if isinstance(d, dict):
        lists = [v for v in d.values() if isinstance(v, list) and v and isinstance(v[0], dict)]
        if lists:
            d = lists[0]
        elif all(isinstance(v, str) for v in d.values()):
            d = [{"id": k, "document": v} for k, v in d.items()]
        elif all(isinstance(v, dict) for v in d.values()):
            d = [{"id": k, **v} for k, v in d.items()]
    return d


DOC_KEYS = ("document", "text", "judgment", "doc", "source")
SUM_KEYS = ("summary", "abstract", "target", "reference")


def load_split(stem, need_summary):
    hits = [p for ext in (".json", ".jsonl") for p in find_files(stem + ext)]
    hits.sort(key=lambda p: not any(k in p.lower() for k in ("summ", "phase_6", "phase6")))
    if not hits:
        print(f"  {stem}: no {stem}.json / {stem}.jsonl under {INPUT_ROOT}")
    for p in hits:
        try:
            d = read_any_json(p)
        except Exception as e:
            print(f"  skip {p}: unreadable ({e.__class__.__name__})"); continue
        if not (isinstance(d, list) and d and isinstance(d[0], dict)):
            print(f"  skip {p}: not a list of records"); continue
        r0 = d[0]
        if any(k in r0 for k in ("label", "labels")):
            print(f"  skip {p}: has label/labels -> Phase 4/5 data"); continue
        dk = next((k for k in DOC_KEYS if k in r0), None)
        sk = next((k for k in SUM_KEYS if k in r0), None)
        ik = next((k for k in ("id", "doc_id", "case_id") if k in r0), None)
        if dk is None or ik is None:
            print(f"  skip {p}: keys {list(r0.keys())}"); continue
        if need_summary and sk is None:
            print(f"  skip {p}: no summary field; keys {list(r0.keys())}"); continue
        rows = [{"id": str(r[ik]), "document": str(r[dk]), "summary": str(r[sk]) if sk else None} for r in d]
        print(f"  {stem}: {p} ({len(rows)} rows)")
        return rows
    return None


MODEL_BIN = find_files("pytorch_model.bin")[0]
MODEL_DIR = os.path.dirname(MODEL_BIN)
print("model dir:", MODEL_DIR, "\ndata:")
train_raw = load_split("train", need_summary=True)
test_raw = load_split("test", need_summary=False)
if train_raw is None or test_raw is None:
    for root, _, files in os.walk(INPUT_ROOT, followlinks=True):
        for f in files:
            print("   ", os.path.join(root, f))
    raise FileNotFoundError("Phase 6 train/test not found - attach the Phase_6_Summ dataset via 'Add Input'.")

In [ ]:
# Phase 1 model + tokenizer
shutil.copy(os.path.join(MODEL_DIR, "model.py"), os.path.join(OUT, "legal_lm_model.py"))
sys.path.insert(0, OUT)
from legal_lm_model import LegalLM, ModelConfig  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

LM_CFG = json.load(open(os.path.join(MODEL_DIR, "config.json")))
BASE_SD = torch.load(MODEL_BIN, map_location="cpu")
tokenizer = Tokenizer.from_file(os.path.join(MODEL_DIR, "tokenizer.json"))
EOS = 0
MAX_POS = LM_CFG["max_seq_len"]
SEP = tokenizer.encode("\n\nSummary:\n").ids
_HDR = {}


def header(i, n):
    k = (i, n)
    if k not in _HDR:
        _HDR[k] = tokenizer.encode(f"Part {i + 1} of {n}:\n").ids
    return _HDR[k]


LEAD_HDR = tokenizer.encode("Judgment (beginning):\n").ids
print("LM config:", LM_CFG)

In [ ]:
# Data: sentence split, chunking, summary-sentence alignment
STOP = set("the of and to in a is that for by was be as on with it this or an are were which has have had not "
           "from at under any such its their his been would shall said also than other into these there".split())


def clean(t):
    t = t.replace("\r", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n\s*\n+", "\n", t)
    return t.strip()


SENT_RE = re.compile(r"(?<=[.!?;])\s+(?=[A-Z0-9(\"'])|\n+")


def sentences(t):
    return [s.strip() for s in SENT_RE.split(clean(t)) if s and s.strip()]


def words(t):
    return re.findall(r"[a-z0-9]+", t.lower())


def content_grams(t):
    w = [x for x in words(t) if len(x) > 2 and x not in STOP]
    return set(w), set(zip(w, w[1:]))


def chunk_doc(doc_text, CH):
    """Greedy sentence packing into chunks of <= CH tokens. Returns list of (token_ids, chunk_text)."""
    sents = sentences(doc_text)
    if not sents:
        sents = [doc_text[:2000] or "empty"]
    enc = tokenizer.encode_batch([(" " if i else "") + s for i, s in enumerate(sents)])
    chunks, cur_ids, cur_txt = [], [], []
    for s, e in zip(sents, enc):
        ids = e.ids
        while len(ids) > CH:                         # very long "sentence": hard split
            if cur_ids:
                chunks.append((cur_ids, " ".join(cur_txt))); cur_ids, cur_txt = [], []
            chunks.append((ids[:CH], tokenizer.decode(ids[:CH]))); ids = ids[CH:]
        if len(cur_ids) + len(ids) > CH and cur_ids:
            chunks.append((cur_ids, " ".join(cur_txt))); cur_ids, cur_txt = [], []
        cur_ids = cur_ids + ids; cur_txt.append(s)
    if cur_ids:
        chunks.append((cur_ids, " ".join(cur_txt)))
    return chunks


def spread(n, k):
    return list(range(n)) if n <= k else sorted(set(np.linspace(0, n - 1, k).round().astype(int).tolist()))


def align(summary, chunk_texts):
    """Assign each summary sentence to the chunk with the highest unigram+bigram recall."""
    grams = [content_grams(t) for t in chunk_texts]
    out = [[] for _ in chunk_texts]
    for s in sentences(summary):
        u, b = content_grams(s)
        if not u:
            if out:
                out[-1].append(s)
            continue
        sc = [len(u & gu) / len(u) + (len(b & gb) / len(b) if b else 0.0) for gu, gb in grams]
        out[int(np.argmax(sc))].append(s)
    return out


def build_doc(r, train=True):
    ch = chunk_doc(r["document"], CFG["chunk_tokens"])
    d = dict(id=r["id"], n_chunks=len(ch), doc_tokens=sum(len(c[0]) for c in ch),
             chunks=[np.asarray(c[0], dtype=np.uint16) for c in ch], summary=r.get("summary"))
    if r.get("summary"):
        assigned = align(r["summary"], [c[1] for c in ch])
        d["targets"] = [np.asarray(tokenizer.encode(" ".join(a)).ids[:CFG["target_max"]] if a else [], dtype=np.uint16)
                        for a in assigned]
        d["ref_parts"] = assigned
        all_ids = np.concatenate(d["chunks"]) if d["chunks"] else np.zeros(1, np.uint16)
        d["lead_in"] = all_ids[:CFG["chunk_tokens"]]
        d["lead_out"] = np.asarray(tokenizer.encode(clean(r["summary"])).ids[:CFG["target_max"]], dtype=np.uint16)
    return d


rng = random.Random(CFG["seed"])
train_rows = [r for r in train_raw if (r["summary"] or "").strip() and (r["document"] or "").strip()]
if len(train_rows) != len(train_raw):
    print(f"  dropped {len(train_raw) - len(train_rows)} training docs with an empty summary/document: "
          f"{[r['id'] for r in train_raw if not ((r['summary'] or '').strip() and (r['document'] or '').strip())][:10]}")
for r in test_raw:
    if not (r["document"] or "").strip():
        r["document"] = "empty document"     # keep the ID so it still gets a prediction
rng.shuffle(train_rows)
if SMOKE:
    train_rows = train_rows[:30]; test_raw = test_raw[:4]
val_rows, trn_rows = train_rows[:CFG["val_docs"]], train_rows[CFG["val_docs"]:]
t0 = time.time()
TRN = [build_doc(r) for r in trn_rows]
VAL = [build_doc(r) for r in val_rows]
TEST = [build_doc(r, train=False) for r in test_raw]
assert len({d["id"] for d in TEST}) == len(TEST), "duplicate test IDs"
del train_raw, train_rows, trn_rows, val_rows
gc.collect()
print(f"built docs in {(time.time() - t0) / 60:.1f} min | train {len(TRN)} | val {len(VAL)} | test {len(TEST)}")
print("chunks/doc p50/p90/max:", *np.percentile([d['n_chunks'] for d in TRN], [50, 90]).astype(int),
      max(d['n_chunks'] for d in TRN), "| test:", *np.percentile([d['n_chunks'] for d in TEST], [50, 90]).astype(int),
      max(d['n_chunks'] for d in TEST))


def chunk_pairs(docs, keep_empty, seed):
    r = random.Random(seed); pairs = []
    for d in docs:
        if "targets" not in d:
            continue
        for i in spread(d["n_chunks"], CFG["max_chunks_train"]):
            tgt = d["targets"][i]
            if len(tgt) == 0 and r.random() > keep_empty:
                continue
            pairs.append((np.concatenate([np.asarray(header(i, d["n_chunks"]), np.uint16), d["chunks"][i],
                                          np.asarray(SEP, np.uint16)]), tgt))
    return pairs


def lead_pairs(docs):
    return [(np.concatenate([np.asarray(LEAD_HDR, np.uint16), d["lead_in"], np.asarray(SEP, np.uint16)]), d["lead_out"])
            for d in docs if "lead_in" in d]


PAIRS ={"chunk": chunk_pairs(TRN, CFG["keep_empty_frac"], 1), "lead": lead_pairs(TRN)}
_vp = chunk_pairs(VAL, CFG["keep_empty_frac"], 2)
VAL_PAIRS = {"chunk": random.Random(3).sample(_vp, min(CFG["val_loss_pairs"], len(_vp))), "lead": lead_pairs(VAL)}
tl = np.array([len(t) for _, t in PAIRS["chunk"]])
print(f"chunk pairs: train {len(PAIRS['chunk'])} (empty targets {np.mean(tl == 0):.1%}, target tokens p50 "
      f"{int(np.median(tl))} p90 {int(np.percentile(tl, 90))}) | val-loss pairs {len(VAL_PAIRS['chunk'])}")
GEN_VAL = [d for d in VAL if "ref_parts" in d][:CFG["val_gen_docs"]]

In [ ]:
# Model wrapper + adaptation methods (LoRA, Houlsby, Pfeiffer, prefix)
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class Adapter(nn.Module):
    def __init__(self, d, b):
        super().__init__()
        self.down, self.up = nn.Linear(d, b), nn.Linear(b, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    def __init__(self, sub, d, b):
        super().__init__()
        self.sub, self.adapter = sub, Adapter(d, b)

    def forward(self, *a, **k):
        return self.adapter(self.sub(*a, **k))


class Prefix(nn.Module):
    """Prefix tuning (Li & Liang): P virtual key/value vectors per layer, produced by an MLP (reparameterisation)."""
    def __init__(self, lm, P, hidden=512):
        super().__init__()
        c = lm.cfg
        self.P, self.L, self.H, self.hd = P, c.n_layer, c.n_head, c.d_model // c.n_head
        idx = torch.randint(1000, c.vocab_size, (P,))
        self.emb = nn.Parameter(lm.tok.weight[idx].detach().clone())
        self.mlp = nn.Sequential(nn.Linear(c.d_model, hidden), nn.Tanh(), nn.Linear(hidden, c.n_layer * 2 * c.d_model))

    def caches(self, B, dtype):
        kv = self.mlp(self.emb).view(self.P, self.L, 2, self.H, self.hd).permute(1, 2, 3, 0, 4)  # L,2,H,P,hd
        kv = kv.to(dtype)
        return [{"k": kv[l, 0][None].expand(B, -1, -1, -1), "v": kv[l, 1][None].expand(B, -1, -1, -1)}
                for l in range(self.L)]


class SummModel(nn.Module):
    def __init__(self, lm):
        super().__init__()
        self.lm = lm
        self.prefix = None

    @property
    def P(self):
        return 0 if self.prefix is None else self.prefix.P

    def hidden(self, ids, positions=None, attn_mask=None, caches=None):
        lm, T = self.lm, ids.shape[1]
        if positions is None:
            cos, sin = lm.rope_cos[:T][None, None], lm.rope_sin[:T][None, None]
        else:
            cos, sin = lm.rope_cos[positions].unsqueeze(1), lm.rope_sin[positions].unsqueeze(1)
        x = lm.tok(ids)
        for i, blk in enumerate(lm.blocks):
            x = blk(x, cos, sin, attn_mask, None if caches is None else caches[i])
        return lm.norm(x)

    def train_hidden(self, ids):
        """Teacher-forced pass over right-padded sequences (prefix keys prepended if present)."""
        if self.prefix is None:
            return self.hidden(ids)
        B, T, P = ids.shape[0], ids.shape[1], self.P
        caches = self.prefix.caches(B, torch.float16 if USE_AMP else torch.float32)
        m = torch.cat([torch.ones(T, P, dtype=torch.bool, device=ids.device),
                       torch.ones(T, T, dtype=torch.bool, device=ids.device).tril()], 1)[None, None]
        return self.hidden(ids, attn_mask=m, caches=caches)


def fresh_lm(layers=None):
    cfg = dict(LM_CFG)
    if layers is not None:
        cfg["n_layer"] = len(layers)
    lm = LegalLM(ModelConfig(**cfg))
    if layers is None:
        lm.load_state_dict(BASE_SD, strict=True)
    else:
        sd = {}
        for k, v in BASE_SD.items():
            if k.startswith("blocks."):
                i = int(k.split(".")[1])
                if i in layers:
                    sd[k.replace(f"blocks.{i}.", f"blocks.{layers.index(i)}.", 1)] = v
            else:
                sd[k] = v
        lm.load_state_dict(sd, strict=True)
    return lm


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False


def build_model(exp):
    m = exp["method"]
    model = SummModel(fresh_lm(exp.get("layers")))
    lm, d = model.lm, LM_CFG["d_model"]
    if m in ("full", "kd"):
        pass
    elif m == "lora":
        freeze(lm)
        for blk in lm.blocks:
            blk.attn.qkv = LoRALinear(blk.attn.qkv, exp["r"], exp["alpha"])
            blk.attn.proj = LoRALinear(blk.attn.proj, exp["r"], exp["alpha"])
            blk.mlp.w12 = LoRALinear(blk.mlp.w12, exp["r"], exp["alpha"])
            blk.mlp.w3 = LoRALinear(blk.mlp.w3, exp["r"], exp["alpha"])
    elif m in ("houlsby", "pfeiffer"):
        freeze(lm)
        for blk in lm.blocks:
            if m == "houlsby":
                blk.attn = WithAdapter(blk.attn, d, exp["bottleneck"])
            blk.mlp = WithAdapter(blk.mlp, d, exp["bottleneck"])
            for p in list(blk.norm1.parameters()) + list(blk.norm2.parameters()):
                p.requires_grad = True
    elif m == "prefix":
        freeze(lm)
        model.prefix = Prefix(lm, exp["n_prefix"])
    else:
        raise ValueError(m)
    return model.to(device)


def param_stats(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

In [ ]:
# Training (teacher-forced CE on summary tokens; optional token-level KD)
def make_batch(pairs):
    seqs, starts = [], []
    for prompt, tgt in pairs:
        s = np.concatenate([prompt, tgt, np.asarray([EOS], np.uint16)]).astype(np.int64)[:MAX_POS + 1]
        seqs.append(s); starts.append(len(prompt))
    T = max(len(s) for s in seqs) - 1
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    tgt = torch.zeros(len(seqs), T, dtype=torch.long)
    mask = torch.zeros(len(seqs), T, dtype=torch.bool)
    for b, (s, st) in enumerate(zip(seqs, starts)):
        n = len(s) - 1
        ids[b, :n] = torch.from_numpy(s[:-1]); tgt[b, :n] = torch.from_numpy(s[1:])
        mask[b, st - 1:n] = True                   # positions that predict summary tokens + EOS
    return ids.to(device), tgt.to(device), mask.to(device)


def target_logits(model, ids, mask):
    h = model.train_hidden(ids)
    return model.lm.head(h[mask]).float()


@torch.no_grad()
def val_loss(model, pairs, bs=8):
    model.eval(); tot, n = 0.0, 0
    for i in range(0, len(pairs), bs):
        ids, tgt, mask = make_batch(pairs[i:i + bs])
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            lg = target_logits(model, ids, mask)
        tot += F.cross_entropy(lg, tgt[mask], reduction="sum").item(); n += int(mask.sum())
    model.train()
    return tot / max(1, n)


def train_model(model, exp, pairs, vpairs, teacher=None):
    params = [p for p in model.parameters() if p.requires_grad]
    dec = [p for p in params if p.dim() >= 2]
    nod = [p for p in params if p.dim() < 2]
    opt = torch.optim.AdamW([dict(params=dec, weight_decay=0.01), dict(params=nod, weight_decay=0.0)],
                            lr=exp["lr"], betas=(0.9, 0.98), eps=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    mb, acc, n = CFG["micro"], CFG["accum"], len(pairs)
    spe = math.ceil(n / (mb * acc))
    sched = dict(total=exp["epochs"] * spe)
    warm = max(3, int(0.05 * sched["total"]))
    budget = exp["minutes"] * 60
    eval_at, t0, step, best, t_meas = set(), time.time(), 0, (float("inf"), None, None), None
    kT, ka = exp.get("T", 2.0), exp.get("alpha", 0.5)

    def lr_mult(s):
        return (s + 1) / warm if s < warm else max(0.05, (sched["total"] - s) / max(1, sched["total"] - warm))

    def snapshot():
        return {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}

    model.train(); done = False
    for ep in range(exp["epochs"]):
        perm = np.random.permutation(n)
        micro = [perm[i:i + mb] for i in range(0, n, mb)]
        for mi in range(0, len(micro), acc):
            grp = micro[mi:mi + acc]
            for g in opt.param_groups:
                g["lr"] = exp["lr"] * lr_mult(step)
            lv = 0.0
            for idx in grp:
                ids, tgt, mask = make_batch([pairs[j] for j in idx])
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                        lg = target_logits(model, ids, mask)
                        loss = F.cross_entropy(lg, tgt[mask])
                        if teacher is not None:
                            with torch.no_grad():
                                tl = target_logits(teacher, ids, mask)
                            kd = F.kl_div(F.log_softmax(lg / kT, -1), F.log_softmax(tl / kT, -1),
                                          reduction="batchmean", log_target=True) * kT ** 2
                            loss = ka * kd + (1 - ka) * loss
                    scaler.scale(loss / len(grp)).backward(); lv += loss.item() / len(grp)
                except torch.cuda.OutOfMemoryError:
                    print("  OOM - skipping micro-batch"); opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if step == 2:
                t_meas = time.time()
            if step == 7:
                sps = (time.time() - t_meas) / 5
                sched["total"] = max(step + 1, min(sched["total"], step + int(max(0, budget - (time.time() - t0)) / sps)))
                eval_at = {max(step + 1, int(round(sched["total"] * f))) for f in np.linspace(0.5, 1.0, CFG["n_evals"])}
                print(f"  {sps:.2f}s/step | planned {exp['epochs'] * spe} -> {sched['total']} steps "
                      f"({sched['total'] * mb * acc / n:.2f} epochs) | evals at {sorted(eval_at)}")
            if step % 25 == 0:
                print(f"  step {step}/{sched['total']} loss {lv:.4f} lr x{lr_mult(step):.2f} {(time.time() - t0) / 60:.1f} min")
            last = step >= sched["total"] or time.time() - t0 > budget * 1.15 or elapsed_h() > CFG["total_hours"] - 0.25
            if step in eval_at or last:
                vl = val_loss(model, vpairs)
                print(f"  [eval step {step}] val loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.1f})")
                if vl < best[0]:
                    best = (vl, step, snapshot())
            if last:
                done = True; break
        if done:
            break
    if best[2] is None:
        best = (val_loss(model, vpairs), step, snapshot())
    model.load_state_dict(best[2])
    print(f"  best val loss {best[0]:.4f} at step {best[1]} | {(time.time() - t0) / 60:.1f} min")
    return dict(train_min=(time.time() - t0) / 60, steps=step, val_loss=best[0])

In [ ]:
# Generation: batched greedy decoding with KV cache, left padding, 3-gram blocking, repetition penalty
@torch.no_grad()
def generate(model, prompts, max_new, min_new=0):
    model.eval()
    lm, B, P = model.lm, len(prompts), model.P
    Tp = max(len(p) for p in prompts)
    max_new = max(1, min(max_new, MAX_POS - Tp))
    ids = torch.zeros(B, Tp, dtype=torch.long)
    valid = torch.zeros(B, Tp, dtype=torch.bool)
    for b, p in enumerate(prompts):
        ids[b, Tp - len(p):] = torch.as_tensor(np.asarray(p, dtype=np.int64)); valid[b, Tp - len(p):] = True
    ids, valid = ids.to(device), valid.to(device)
    pos = (valid.long().cumsum(1) - 1).clamp(min=0)
    dt = torch.float16 if USE_AMP else torch.float32
    caches = model.prefix.caches(B, dt) if model.prefix is not None else [dict() for _ in lm.blocks]
    eye = torch.eye(Tp, dtype=torch.bool, device=device)[None]
    m = (torch.ones(Tp, Tp, dtype=torch.bool, device=device).tril()[None] & valid[:, None, :]) | eye
    if P:
        m = torch.cat([torch.ones(B, Tp, P, dtype=torch.bool, device=device), m], -1)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        h = model.hidden(ids, positions=pos, attn_mask=m[:, None], caches=caches)
        logits = lm.head(h[:, -1]).float()
    key_valid = torch.cat([torch.ones(B, P, dtype=torch.bool, device=device), valid], 1)
    cur = pos[:, -1].clone()
    V = logits.shape[-1]
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    gen = [[] for _ in range(B)]
    ngr = [dict() for _ in range(B)]
    done = np.zeros(B, dtype=bool)
    n = CFG["no_repeat_ngram"]
    for step in range(max_new):
        rp = CFG["rep_penalty"]
        logits = torch.where(seen, torch.where(logits > 0, logits / rp, logits * rp), logits)
        if step < min_new:
            logits[:, EOS] = -1e9
        ban_b, ban_t = [], []
        for b in range(B):
            if not done[b] and len(gen[b]) >= n - 1:
                for t in ngr[b].get(tuple(gen[b][-(n - 1):]), ()):
                    ban_b.append(b); ban_t.append(t)
        if ban_b:
            logits[torch.tensor(ban_b, device=device), torch.tensor(ban_t, device=device)] = -1e9
        nxt = logits.argmax(-1)
        nxt_c = nxt.cpu().numpy()
        for b in range(B):
            if done[b]:
                nxt_c[b] = EOS; continue
            t = int(nxt_c[b])
            if t == EOS:
                done[b] = True; continue
            gen[b].append(t)
            if len(gen[b]) >= n:
                ngr[b].setdefault(tuple(gen[b][-n:-1]), set()).add(gen[b][-1])
        if done.all():
            break
        nxt = torch.as_tensor(nxt_c, device=device)
        seen[torch.arange(B, device=device), nxt] = True
        key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], 1)
        cur = (cur + 1).clamp(max=MAX_POS - 1)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            h = model.hidden(nxt[:, None], positions=cur[:, None], attn_mask=key_valid[:, None, None, :], caches=caches)
            logits = lm.head(h[:, -1]).float()
    return gen


def decode(ids):
    return clean(tokenizer.decode([int(i) for i in ids])) if ids else ""


def postprocess(pieces):
    """Stitch chunk summaries in order, drop near-duplicate sentences (redundancy reduction) and fragments."""
    kept, kept_sets = [], []
    for s in sentences(" ".join(p for p in pieces if p)):
        w = set(words(s))
        if len(w) < 3:
            continue
        if any(len(w & k) / max(1, len(w | k)) >= 0.7 for k in kept_sets):
            continue
        kept.append(s); kept_sets.append(w)
    return " ".join(kept)


def summarize(model, docs, mode="chunk"):
    t0 = time.time()
    jobs = []                                   # (doc_index, chunk_index, prompt)
    for di, d in enumerate(docs):
        if mode == "lead":
            all_ids = np.concatenate(d["chunks"])[:CFG["chunk_tokens"]]
            jobs.append((di, 0, np.concatenate([np.asarray(LEAD_HDR, np.uint16), all_ids, np.asarray(SEP, np.uint16)])))
        else:
            for i in spread(d["n_chunks"], CFG["max_chunks_infer"]):
                jobs.append((di, i, np.concatenate([np.asarray(header(i, d["n_chunks"]), np.uint16), d["chunks"][i],
                                                    np.asarray(SEP, np.uint16)])))
    order = np.argsort([len(j[2]) for j in jobs])
    out = {}
    max_new = CFG["target_max"] if mode == "lead" else CFG["max_new"]
    for s in range(0, len(order), CFG["gen_batch"]):
        bj = [jobs[k] for k in order[s:s + CFG["gen_batch"]]]
        for (di, ci, _), g in zip(bj, generate(model, [j[2] for j in bj], max_new)):
            out[(di, ci)] = decode(g)
    summaries = []
    for di, d in enumerate(docs):
        pieces = [out[k] for k in sorted(k for k in out if k[0] == di)]
        sm = postprocess(pieces)
        if not sm:                              # never submit an empty summary
            p = jobs[[j[0] for j in jobs].index(di)][2]
            sm = decode(generate(model, [p], 120, min_new=60)[0]) or "The appeal was disposed of."
        summaries.append(sm)
    return summaries, (time.time() - t0) / max(1, len(docs))

In [ ]:
# Metrics: ROUGE-1 / ROUGE-L F1 (bit-parallel LCS), coverage by document section, redundancy, length
def lcs(a, b):
    if not a or not b:
        return 0
    pos = {}
    for i, t in enumerate(a):
        pos[t] = pos.get(t, 0) | (1 << i)
    nbit = len(a); mask = (1 << nbit) - 1; Vb = mask
    for t in b:
        U = Vb & pos.get(t, 0)
        Vb = ((Vb + U) | (Vb - U)) & mask
    return nbit - bin(Vb).count("1")


def prf(overlap, n_pred, n_ref):
    p, r = overlap / max(1, n_pred), overlap / max(1, n_ref)
    return 2 * p * r / max(1e-9, p + r), r


def score_summaries(preds, docs):
    r1, rl, cov, red, lens = [], [], {"begin": [], "middle": [], "end": []}, [], []
    for pred, d in zip(preds, docs):
        pw, rw = words(pred), words(d["summary"])
        f1, _ = prf(sum((Counter(pw) & Counter(rw)).values()), len(pw), len(rw)); r1.append(f1)
        rl.append(prf(lcs(rw, pw), len(pw), len(rw))[0])
        pc = Counter(pw)
        nch = max(1, d["n_chunks"])
        for i, part in enumerate(d["ref_parts"]):          # recall of reference content by document section
            if not part:
                continue
            sec = "begin" if i < nch / 3 else ("middle" if i < 2 * nch / 3 else "end")
            rc = Counter(words(" ".join(part)))
            cov[sec].append(sum((rc & pc).values()) / max(1, sum(rc.values())))
        tri = list(zip(pw, pw[1:], pw[2:]))
        red.append(1 - len(set(tri)) / max(1, len(tri)))
        lens.append((len(pw), len(rw)))
    out = dict(val_rouge1_f=round(float(np.mean(r1)), 4), val_rougeL_f=round(float(np.mean(rl)), 4),
               coverage_begin=round(float(np.mean(cov["begin"])), 4) if cov["begin"] else None,
               coverage_middle=round(float(np.mean(cov["middle"])), 4) if cov["middle"] else None,
               coverage_end=round(float(np.mean(cov["end"])), 4) if cov["end"] else None,
               redundancy_trigram=round(float(np.mean(red)), 4),
               avg_words_pred=int(np.mean([a for a, _ in lens])), avg_words_ref=int(np.mean([b for _, b in lens])))
    if USE_BERTSCORE:
        try:
            from bert_score import score as bscore
            _, _, Fb = bscore(preds, [d["summary"] for d in docs], lang="en", verbose=False)
            out["val_bertscore_f"] = round(float(Fb.mean()), 4)
        except Exception as e:
            print("  BERTScore unavailable:", e)
    return out

In [ ]:
# Outputs
RESULTS, VAL_SUMS, TEST_SUMS = [], {}, {}


def write_submission(name, sums):
    sub = [{"id": d["id"], "summary": s} for d, s in zip(TEST, sums)]
    assert len(sub) == len(TEST) and all(isinstance(x["summary"], str) and x["summary"] for x in sub)
    tag = ("smoke_" if SMOKE else "") + name
    jp = os.path.join(OUT, f"{tag}_prediction.json")
    json.dump(sub, open(jp, "w"), ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(OUT, f"{tag}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    print(f"  -> {tag}.zip (avg {np.mean([len(s.split()) for s in sums]):.0f} words/summary)")


def record(name, method, val_sums, test_sums, extra):
    m = score_summaries(val_sums, GEN_VAL)
    row = dict(name=name, method=method, **m, **extra)
    RESULTS.append(row); VAL_SUMS[name], TEST_SUMS[name] = val_sums, test_sums
    json.dump([{"id": d["id"], "summary": s, "reference": d["summary"]} for d, s in zip(GEN_VAL, val_sums)],
              open(os.path.join(OUT, f"val_summaries_{name}.json"), "w"), ensure_ascii=False)
    write_submission(name, test_sums)
    with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
        keys = list(dict.fromkeys(k for r in RESULTS for k in r))
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(RESULTS)
    print("  ", row)


def run_experiment(exp, teacher=None):
    print("\n" + "=" * 90 + f"\n{exp['name']} ({exp['method']}) | notebook time {elapsed_h():.2f} h\n" + "=" * 90)
    seed_all(CFG["seed"])
    mode = exp.get("mode", "chunk")
    model = build_model(exp)
    total, trainable = param_stats(model)
    print(f"  params: total {total / 1e6:.2f}M | trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")
    if USE_AMP:
        torch.cuda.reset_peak_memory_stats()
    info = train_model(model, exp, PAIRS[mode], VAL_PAIRS[mode], teacher)
    peak = torch.cuda.max_memory_allocated() / 2 ** 30 if USE_AMP else 0.0
    vs, s_val = summarize(model, GEN_VAL, mode)
    ts, s_test = summarize(model, TEST, mode)
    extra = dict(val_loss=round(info["val_loss"], 4), total_params_M=round(total / 1e6, 2),
                 trainable_params_M=round(trainable / 1e6, 3), trainable_pct=round(100 * trainable / total, 2),
                 size_fp16_MB=round(total * 2 / 2 ** 20, 1), peak_train_mem_GB=round(peak, 2),
                 train_min=round(info["train_min"], 1), steps=info["steps"],
                 gen_sec_per_doc=round((s_val + s_test) / 2, 2), long_doc=mode)
    record(exp["name"], exp["method"], vs, ts, extra)
    return model

In [ ]:
# 01 — Full fine-tuning (reference + teacher) and length analysis
E = {e["name"]: e for e in EXPS}
teacher = run_experiment(E["01_fullft_chunk"])

rows = []
for frac in (0.5, 0.75, 1.0):                   # effect of generation length on ROUGE (post-hoc truncation)
    cut = [" ".join(s.split()[:max(1, int(len(s.split()) * frac))]) for s in VAL_SUMS["01_fullft_chunk"]]
    rows.append(dict(length_fraction=frac, **{k: v for k, v in score_summaries(cut, GEN_VAL).items()
                                               if k.startswith(("val_", "avg_"))}))
    print("  length analysis:", rows[-1])
with open(os.path.join(OUT, "length_analysis_01.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

In [ ]:
# 06 — Knowledge distillation (12L full-FT teacher -> 6L student initialised from Phase 1 layers)
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()
m = run_experiment(E["06_kd_student6"], teacher=teacher)
del m, teacher; free()

In [ ]:
# 02 — LoRA
m = run_experiment(E["02_lora_chunk"]); del m; free()

In [ ]:
# 03 — Houlsby adapters
m = run_experiment(E["03_houlsby_adapter"]); del m; free()

In [ ]:
# 04 — Pfeiffer adapters
m = run_experiment(E["04_pfeiffer_adapter"]); del m; free()

In [ ]:
# 05 — Prefix tuning
m = run_experiment(E["05_prefix_tuning"]); del m; free()

In [ ]:
# 07 — Flat baseline without chunking (skipped if out of time)
e7 = E["07_lora_lead"]
if elapsed_h() + (e7["minutes"] + 10) / 60 < CFG["total_hours"]:
    m = run_experiment(e7); del m; free()
else:
    print(f"Skipping 07 — {elapsed_h():.2f} h used")

In [ ]:
# 08 — MBR consensus: per document pick the candidate with the highest mean ROUGE-1 F1 against the others
def rouge1_f(a, b):
    wa, wb = words(a), words(b)
    return prf(sum((Counter(wa) & Counter(wb)).values()), len(wa), len(wb))[0]


def mbr(cands_per_doc):
    out, picks = [], []
    for cands in cands_per_doc:
        sc = [np.mean([rouge1_f(c, o) for j, o in enumerate(cands) if j != i]) if len(cands) > 1 else 0
              for i, c in enumerate(cands)]
        k = int(np.argmax(sc)); out.append(cands[k]); picks.append(k)
    return out, picks


pool = [n for n in VAL_SUMS if n.startswith(("01", "02", "03", "04", "05", "06"))]
vs, vp = mbr(list(zip(*[VAL_SUMS[n] for n in pool])))
ts, tp = mbr(list(zip(*[TEST_SUMS[n] for n in pool])))
print("\n08_mbr_consensus over", pool, "| test picks:", dict(Counter(pool[k] for k in tp)))
record("08_mbr_consensus", "mbr(" + "+".join(n[:2] for n in pool) + ")", vs, ts, dict(long_doc="chunk"))

In [ ]:
# Summary table
print(f"\nTotal notebook time {elapsed_h():.2f} h")
cols = ["name", "val_rouge1_f", "val_rougeL_f", "coverage_begin", "coverage_middle", "coverage_end",
        "redundancy_trigram", "avg_words_pred", "avg_words_ref", "trainable_params_M", "peak_train_mem_GB",
        "train_min", "gen_sec_per_doc"]
print(" | ".join(cols))
for r in RESULTS:
    print(" | ".join(str(r.get(c, "")) for c in cols))

In [ ]:
# Download links — one bundle with every submission + reports
from IPython.display import FileLink, display, HTML  # noqa: E402

subs = sorted(f for f in os.listdir(OUT) if f.endswith(".zip") and not f.startswith("ALL_"))
bundle = os.path.join(OUT, "ALL_submissions_and_reports.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in subs:
        z.write(os.path.join(OUT, f), arcname=f"submissions/{f}")
    for f in sorted(os.listdir(OUT)):
        if f.endswith(".csv") or f.startswith("val_summaries_"):
            z.write(os.path.join(OUT, f), arcname=f"reports/{f}")
print(f"Bundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB), {len(subs)} submissions")
os.chdir(OUT)
display(HTML("<b>Click to download:</b>"))
display(FileLink(os.path.basename(bundle)))
for f in subs:
    display(FileLink(f))